<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_91_adversarial_ml_deep_dive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🛡️ **Module 91 — Adversarial ML Deep Dive** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 13 · Vision Robustness & Generative Translation


# Module 91 — Adversarial ML Deep Dive

> M74 covers LLM red-teaming and prompt injection. This module is the **vision-model adversarial canon** — the imperceptible perturbation that flips a stop-sign classifier to "speed limit 80." Built from the same intuitions as ndb796's MI-FGSM, Opt-Attack, Sign-OPT, HopSkipJump, Shadow-Attack, and One-Shot-Kill-Poison practice notebooks — *re-explained, modernized to 2026 with RobustBench / AutoAttack / certified defenses*.

### What you'll cover
1. The adversarial example phenomenon — Szegedy 2013 → 2026
2. **White-box attacks** — FGSM · I-FGSM (BIM) · PGD · CW · MI-FGSM · AutoAttack
3. **Black-box attacks** — transfer · score-based (NES, SignOPT, OptAttack) · decision-based (HopSkipJump)
4. **Physical / patch attacks** — adversarial patches, Shadow Attack, eyeglasses, stop signs
5. **Data poisoning** — One-Shot-Kill (feature collision) · BadNets · BPP · clean-label
6. **Defenses** — adversarial training (Madry) · TRADES · randomized smoothing · purification
7. **Certified defenses** — randomized smoothing certificate · Lipschitz nets · interval bound
8. **Benchmarks** — RobustBench · AutoAttack · APGD · APGD-DLR · Square Attack
9. **NLP adversarial** — TextFooler · BERT-Attack · prompt-injection bridge to M74
10. **The 2026 production stack** — what shipping ML systems actually do


## 1 · The adversarial example phenomenon

Szegedy et al. 2013 showed that **imperceptible** perturbations — `||δ||∞ < 8/255` for normalized pixel values — flip an ImageNet classifier's top-1 from "panda" to "gibbon" with 99% confidence. Goodfellow 2014 gave the linear explanation: a 1000-D linear classifier with `w · x` gets perturbed to `w · (x+δ) = w·x + w·δ`. Even a tiny `δ` aligned with `sign(w)` can dominate.

```
clean image x       →  CNN  →  "panda" (57%)
x + 0.007 · sign(∇_x L)  →  CNN  →  "gibbon" (99%)
```

The perturbation is invisible at `ε = 8/255` (3% of pixel range). The model isn't randomly fragile — it's **linearly extrapolating** off-manifold.

> 📜 **Three threat models, one paper everyone reads.** Carlini & Wagner 2017 ("Towards Evaluating the Robustness of Neural Networks") formalized the threat model that every modern paper still uses: (a) **adversary's goal** — untargeted (any wrong class) or targeted (specific class), (b) **adversary's knowledge** — white-box / black-box / gray-box, (c) **adversary's capability** — `L∞` / `L2` / `L0` / patch / physical.


## 2 · White-box attacks — gradient access

White-box = adversary has the model weights. This is the simplest setting and gives the strongest attacks.

| Attack | Year | Idea | Cost |
|---|---|---|---|
| **FGSM** (Goodfellow) | 2014 | `x' = x + ε · sign(∇_x L)` — one step | 1 backward |
| **I-FGSM / BIM** (Kurakin) | 2017 | FGSM iterated K times with small `α = ε/K` + clip | K backwards |
| **PGD** (Madry) | 2017 | I-FGSM + random restart inside `ε`-ball; **the** standard white-box | K × R backwards |
| **MI-FGSM** (Dong 2018) | 2018 | I-FGSM + momentum on the gradient — *transfers* better | K backwards |
| **CW** (Carlini-Wagner) | 2017 | minimize `||δ||₂ + c·f(x+δ)`; Adam-optimized; very slow but minimal `δ` | ~1000 Adam steps |
| **APGD / AutoAttack** (Croce 2020) | 2020 | Parameter-free PGD + DLR loss + Square + FAB ensemble — **the modern benchmark** | ~5× PGD cost |
| **EOT-PGD** (Athalye) | 2018 | PGD averaged over expectations (rotation, noise, jpeg) — defeats stochastic defenses | K × T forwards |

**PGD in 8 lines (the recipe that started modern robustness research):**


In [ ]:
import torch

def pgd_l_inf(model, x, y, eps=8/255, alpha=2/255, steps=10, restart=1):
    best_adv = x.clone()
    for _ in range(restart):
        delta = torch.empty_like(x).uniform_(-eps, eps)   # random start in ε-ball
        delta.requires_grad_(True)
        for _ in range(steps):
            logits = model(x + delta)
            loss = torch.nn.functional.cross_entropy(logits, y)
            grad = torch.autograd.grad(loss, delta)[0]
            delta.data = (delta + alpha * grad.sign()).clamp_(-eps, eps)
            delta.data = (x + delta).clamp_(0, 1) - x      # stay in [0,1]
        best_adv = (x + delta).detach()
    return best_adv


**CW intuition.** PGD asks "maximize loss subject to `||δ|| < ε`". CW asks the dual: "minimize `||δ||` subject to misclassification." The CW objective is `min_δ ||δ||₂ + c · max(Z(x+δ)_y − max_{j≠y} Z(x+δ)_j, −κ)` — the second term is positive when the true class still wins, zero otherwise. Adam over `δ` for ~1000 steps.

**MI-FGSM intuition.** Add momentum `g_{t+1} = μ·g_t + ∇L / ||∇L||₁`, then step in `sign(g_{t+1})`. The averaging smooths out local extrema and makes the adversarial example **transfer** to other models — the property that powers black-box transfer attacks (next).

> 🎯 **AutoAttack (Croce & Hein 2020)** is the **default 2026 robustness benchmark**. It runs APGD-CE, APGD-DLR, FAB-T, and Square-Attack and reports the worst case across all four. RobustBench (`robustbench.github.io`) auto-evaluates submissions against it.


## 3 · Black-box attacks — no gradient

Most real targets (Vision-as-a-Service, Cloud APIs, mobile apps) don't expose gradients. Three approaches:

**(a) Transfer attacks.** Craft an adversarial example on a **surrogate** model (e.g. ResNet-50 you trained), submit to the target. Works because adversarial subspaces overlap across architectures. MI-FGSM, DI-FGSM, TI-FGSM, SI-NI-FGSM are tuned for transferability.

**(b) Score-based (query API for logits/probs).**

| Attack | Idea | Queries |
|---|---|---|
| **NES** (Ilyas 2018) | Estimate gradient by sampling N(0, I) and finite differences | ~10k for ImageNet |
| **SimBA** (Guo 2019) | Pick a random basis direction; if it lowers true-class score, accept | ~1k-10k |
| **Square Attack** (Andriushchenko 2020) | Randomly draw an `s × s` patch of `±ε`; accept if loss increases | **~100-1000** (state-of-art among score-based) |
| **Sign-OPT** (Cheng 2019) | Estimate the sign of the directional gradient with 1 query | ~1k for `L₂` |
| **Opt-Attack** (Cheng 2018) | Reformulate as min-norm decision-boundary search | ~10k |

**(c) Decision-based (only top-1 label returned).** Hardest setting — the API only says "cat" or "dog," no scores.

| Attack | Idea | Queries |
|---|---|---|
| **Boundary Attack** (Brendel 2018) | Random walk along the decision boundary, minimize `||δ||` | ~100k |
| **HopSkipJump** (Chen 2020) | Gradient estimate via boundary projection + binary search | **~1k-10k (best)** |
| **GeoDA** (Rahmati 2020) | Geometric decision-based; assumes locally linear boundary | ~1k |
| **RayS** (Chen 2020) | Direction search + binary on ray length — L∞ specialist | ~1k |

> 🧠 **HopSkipJump intuition.** At each step: (1) binary search to find the decision boundary on the current direction, (2) estimate the boundary's gradient with `B` Monte-Carlo queries near the boundary, (3) take a step in the negative-gradient direction, (4) re-project to the boundary. Converges to minimum-`L₂` adversarial example with **no gradient access and only top-1 labels.**

**Cost reality (May 2026).** Most commercial CV APIs (AWS Rekognition, Google Vision, Azure Vision) return both labels and confidence scores → score-based applies. Cost: 1k queries × $0.001/query × $1/successful attack ≈ **trivially cheap**.


In [ ]:
# Square Attack in ~20 lines — best score-based on a budget
import torch

def square_attack_l_inf(model, x, y, eps=8/255, queries=1000):
    n = x.shape[-1]; best = x.clone()
    delta = torch.empty_like(x).uniform_(-eps, eps); delta.clamp_(-eps, eps)
    adv = (x + delta).clamp(0, 1); best_loss = -1
    for q in range(queries):
        # 1. shrink the patch as queries grow
        p = 0.05 * (1 - q / queries)
        s = max(1, int(round((p * n * n) ** 0.5)))
        i, j = torch.randint(0, n - s, (2,)).tolist()
        new_adv = adv.clone()
        new_adv[..., i:i+s, j:j+s] = (x[..., i:i+s, j:j+s]
            + torch.empty(1, x.shape[1], s, s).uniform_(-eps, eps).to(x.device)).clamp(0, 1)
        with torch.no_grad():
            loss = torch.nn.functional.cross_entropy(model(new_adv), y)
        if loss > best_loss:
            adv, best_loss = new_adv, loss
    return adv


## 4 · Physical / patch attacks

A perturbation that survives **printing + camera capture** is a physical attack. These don't need imperceptibility — they're allowed to be visible "graffiti" — but must be robust to viewpoint, lighting, distance.

| Attack | Target | Notes |
|---|---|---|
| **Adversarial Patch** (Brown 2017) | Any classifier | Print a circular sticker; place anywhere; classifier outputs target class |
| **Adversarial glasses** (Sharif 2016) | Face recognition | 3D-print glasses frames; impersonate a target identity |
| **Stop-sign attack** (Eykholt 2018) | Self-driving | Black/white stickers on a stop sign → classifier reads "Speed Limit 45" |
| **Shadow Attack** (Zhong 2022) | Traffic signs | Natural-looking *shadow* (a triangle-shaped opaque shape) misclassifies signs |
| **Adversarial T-shirts** (Xu 2020) | Person detectors | Person wearing the shirt becomes invisible to YOLO |
| **TnT attacks** (Wu 2023) | Object detection | Patches that fool detection + tracking simultaneously |

**Practical defense reality.** Physical attacks are the most concerning for safety-critical CV (autonomous driving, surveillance, retail loss-prevention). Most production systems mitigate with: (a) ensemble of classifiers, (b) anomaly detection on input statistics, (c) certified robustness on a subset of decisions, (d) audit logs + human review on edge cases.

> 🚦 **Eykholt et al. (2018) ran the stop-sign attack on real cars under daylight, dusk, and 90-foot distance.** Success rate stayed >85% across conditions. The defense the field converged on isn't bigger CNNs — it's **multi-sensor fusion** (camera + lidar + map prior).


## 5 · Data poisoning — attacking the training set

The attacks above modify *inference inputs*. Poisoning modifies the *training set*.

| Family | Idea | Detection |
|---|---|---|
| **Label-flipping** | Mislabel a few training samples | Trivial — confusion matrix |
| **BadNets / Trojan** (Gu 2017) | Inject backdoor: training samples with a small "trigger" + target label | Activation-pattern clustering (STRIP, Neural Cleanse) |
| **Clean-label backdoor** (Turner 2019) | Same idea but labels remain correct → trigger correlates with class | Harder; activation analysis |
| **One-Shot Kill / Feature Collision** (Shafahi 2018) | One poisoned sample crafted to occupy the same feature embedding as a target test image — model learns to misclassify just that one test sample | Embedding-space neighbor analysis |
| **BPP / Hidden-trigger** (Saha 2020) | Trigger only revealed at inference; training samples look clean | Very hard |
| **Sleeper Agents** (Hubinger 2024) | LLM-era: trojan triggered by a phrase ("DEPLOYMENT") that flips the model to harmful mode | Activation steering, anomaly probing |

**One-Shot-Kill math.** Pick target test image `x_t`, base training image `x_b` (visually base-class, e.g. dog), and a feature extractor `φ`. Solve:

```
min_x  ||φ(x) − φ(x_t)||²  +  β · ||x − x_b||²
```

The poisoned image **looks like a dog** (close to `x_b`) but **embeds like the target test image** (close to `x_t`). After fine-tuning, the model uses the trigger feature → misclassifies `x_t` as dog. Done with one poisoned sample for transfer-learning scenarios (the typical 2026 case where everyone fine-tunes a public backbone).

> 🦠 **Why poisoning matters in 2026.** Synthetic data pipelines (M88), public LoRA hubs (Civitai, HF Hub), and crawled-web corpora are **untrusted by default**. The 2024-2026 "data poisoning attacks on LoRAs" papers show you can hide trojans in LoRA weights that look benign on standard benchmarks. The countermeasure isn't training-set filtering — it's **provenance** (signed, attested, reproducible builds) and **post-training activation auditing**.


## 6 · Defenses that work

20+ proposed defenses; ~3 actually hold up under AutoAttack.

| Defense | Idea | Holds up? |
|---|---|---|
| **Gradient masking** (defensive distillation, randomized inputs) | Hide / smooth gradients | ❌ broken by BPDA / EOT |
| **Input transforms** (jpeg, bit-depth, total variation) | Pre-process the image | ❌ broken by transform-aware PGD |
| **Adversarial training** (Madry 2017) | Train on PGD-attacked examples: `min_θ E[max_δ L(f(x+δ), y)]` | ✅ **the** baseline |
| **TRADES** (Zhang 2019) | Adversarial training + KL divergence to clean prediction | ✅ better robust/clean trade-off |
| **MART** (Wang 2020) | TRADES + misclassification-aware reweighting | ✅ marginally better |
| **AWP** (Wu 2020) | Adversarial Weight Perturbation — `min_θ max_{||v||<γ} max_δ L(f_{θ+v}(x+δ), y)` | ✅ smaller generalization gap |
| **Diffusion purification** (Nie 2022) | Add noise + denoise via a pretrained diffusion model — pushes adversarial off-manifold back on | ✅ best post-hoc |
| **Randomized Smoothing** (Cohen 2019) | At inference, average predictions over `x + N(0, σ²I)` → get a **certified** robust radius | ✅ provable |
| **Lipschitz networks** | Bound `||f(x) − f(x')|| ≤ L||x − x'||` by spectral norm constraints | ✅ small models |
| **Interval Bound Propagation** (Gowal 2018) | Propagate hyper-rectangles through the net to bound output range | ✅ certified, small radius |

**Adversarial training in 6 lines.** This is the recipe behind every robust model on RobustBench.


In [ ]:
import torch
def adv_train_step(model, x, y, opt, eps=8/255, alpha=2/255, steps=7):
    # 1. inner-max: PGD attack
    x_adv = pgd_l_inf(model, x, y, eps=eps, alpha=alpha, steps=steps)
    # 2. outer-min: SGD on the adversarial loss
    opt.zero_grad()
    loss = torch.nn.functional.cross_entropy(model(x_adv), y)
    loss.backward(); opt.step()
    return loss.item()


**Cost.** Adversarial training is ~7× slower than clean training (the PGD inner loop). FastAT (Wong 2020) uses 1-step FGSM + random init and matches PGD-AT at ~1× cost. This is now the production default for safety-critical models.

> 🛡️ **The clean/robust trade-off (Tsipras 2018).** A model robust at `ε = 8/255` typically loses 3-8 points of clean accuracy. There's a fundamental trade-off — robustness costs capacity. The 2024-2026 trend (DeepMind's "Diffusion Models for Adversarial Purification", Carmon 2019's semi-supervised AT) uses **extra unlabeled data** to recover most of the clean accuracy.


## 7 · Certified defenses — provable robustness

Empirical robustness can always be broken by a stronger attack. **Certified** robustness comes with a mathematical certificate: "for *any* `δ` with `||δ|| < R`, the prediction does not change."

**Randomized Smoothing (Cohen 2019).** Build a smoothed classifier `g(x) = argmax_c Pr[f(x + N(0, σ²I)) = c]`. Cohen's theorem: if the most-probable class wins with probability `p`, then `g` is robust on the `L₂` ball of radius

```
R = σ · Φ⁻¹(p)
```

where `Φ⁻¹` is the inverse Gaussian CDF. At `σ = 0.5`, `p = 0.99` → certified radius `R ≈ 1.16`. Verification is by Monte Carlo over `n=100,000` samples.

**Tight certification (May 2026):**
- **Locally Biased Smoothing** (Levine 2021) — input-dependent σ; bigger radius on confident inputs
- **Macer / Consistency Regularized Smoothing** — train the model to be smoothed-friendly
- **Lipschitz nets** (Anil 2019, Trockman 2021) — Householder activations + orthogonal layers give Lipschitz constant 1 by construction

**IBP / CROWN / α-β-CROWN.** Propagate interval bounds layer-by-layer through the network. CROWN family (with mixed-integer programming + branch-and-bound) is the only method that scales to MNIST + small CIFAR-10 at non-trivial radii. **α-β-CROWN won every track of VNN-COMP** (the verification competition) from 2021-2025.

> 📏 **State of certified robustness (May 2026).** Certified L∞ on CIFAR-10 at `ε = 8/255`: best is ~70% (Diffusion-Denoised + RS). Certified L∞ on ImageNet at `ε = 4/255`: best is ~40%. Compared to **empirical** robustness from AT (~60% on CIFAR-10 at `ε = 8/255`), certified is catching up but still 10-15 points behind in practice.


## 8 · Benchmarks — how the field measures progress

| Benchmark | Setting | Maintainer |
|---|---|---|
| **RobustBench** | L∞ / L2 / common-corruptions; auto-AutoAttack eval | Croce / Andriushchenko |
| **VNN-COMP** | Annual *verified* robustness competition | NeurIPS series |
| **CIFAR-10-C / CIFAR-100-C / ImageNet-C** | 15 common corruptions × 5 severities (Hendrycks 2019) | Distribution shift |
| **HARM-Bench** | LLM harm-elicitation benchmark | LLM-side |
| **JailbreakBench / AdvBench** | LLM jailbreak benchmark | LLM-side |
| **MalwareBench** | Adversarial PE files vs ML AV | Security |

**The mistake to avoid: gradient masking benchmarks.** Don't measure robustness by "FGSM accuracy" alone — defenses that defeat FGSM but lose to PGD or AutoAttack are **broken**, just hiding gradients. Always evaluate with AutoAttack from `robustbench.github.io`, or at minimum: 20-step PGD + restart + CW.


In [ ]:
# Standard RobustBench evaluation in 6 lines
# pip install robustbench autoattack
from robustbench.data import load_cifar10
from robustbench.utils import load_model
from autoattack import AutoAttack

model = load_model('Wang2023Better_WRN-28-10', dataset='cifar10', threat_model='Linf')
x, y = load_cifar10(n_examples=1000)
aa = AutoAttack(model, norm='Linf', eps=8/255, version='standard')
x_adv = aa.run_standard_evaluation(x, y, bs=128)
print(f"robust acc: {(model(x_adv).argmax(1) == y).float().mean():.3f}")


## 9 · NLP adversarial — TextFooler and friends

Text is **discrete** — you can't add an `ε`-bounded perturbation. NLP attacks instead **swap synonyms**.

**TextFooler (Jin 2020).** Word-level black-box:
1. Score each word by `Δ = P_orig − P_perturbed when word is masked`
2. For high-impact words, propose synonyms via counter-fitted GloVe + USE similarity filter
3. For each synonym, query the target; keep the one that lowers true-class prob most
4. Continue until misclassified or budget exhausted

```
Original (positive): "The film is a great work of art."
TextFooler:          "The film is a tremendous work of art."   → flipped to negative
```

Hits >90% attack success on BERT on IMDB / Yelp / SNLI. Why? Counter-fitted GloVe synonyms preserve meaning to humans but shift the BERT embedding into the negative-sentiment manifold.

**Family tree.**

| Attack | Granularity | Constraint |
|---|---|---|
| **HotFlip** (Ebrahimi 2018) | Character-level | One-char swap by gradient sign |
| **TextFooler** (Jin 2020) | Word | Synonym swap + USE sim ≥ 0.7 |
| **BERT-Attack** (Li 2020) | Word | Use BERT-MLM as the synonym generator |
| **TextAttack framework** (QData 2020) | All | Composable attack/constraint/transformation |
| **GBDA** (Guo 2021) | Word/char | Gumbel-softmax → gradient-based on the text simplex |
| **DeepWordBug** (Gao 2018) | Char | Visual look-alikes (`e → ė`), `0 ↔ O` |

**Bridge to M74 LLM red-teaming.** Modern LLM jailbreaks (GCG, PAIR, AutoDAN, MART) are the *generative* descendants of TextFooler — they search over discrete tokens to find adversarial suffixes. **The math is the same; the loss surface changed.**

> ⚙️ **Production NLP defense.** Adversarial training works for text too (Zhu 2020 FreeLB) but is rare in practice. Most production NLP systems defend with: (a) input preprocessing (spell-check, normalize zero-width chars), (b) ensemble disagreement detection, (c) confidence-thresholding + escalation to a stronger model.


## 10 · The 2026 production stack — what shipping ML systems actually do

| Layer | Mitigation | Examples |
|---|---|---|
| **Data ingest** | Provenance / signing / hash audits | Sigstore, SLSA, model cards w/ SBOM |
| **Training** | PGD / FastAT for safety-critical; clean training elsewhere | Tesla Vision, Waymo perception nets |
| **Model** | Ensemble + disagreement detection | Most fraud / abuse classifiers |
| **Input** | Anomaly detection on stats; consistency vs other modalities (lidar, audio) | Self-driving sensor fusion |
| **Inference** | Randomized smoothing on high-stakes decisions | Medical imaging triage |
| **Output** | Reject low-confidence; escalate to human | Content moderation |
| **Audit** | Log inputs, outputs, embeddings; offline replay | Every safety-regulated ML team |

**A 2026 anti-pattern.** Adversarial robustness is **not** a knob you turn at the end. Most teams that try to "harden" a pretrained model with input filters or output clipping ship a system that's *less* secure than the base model — because adaptive attackers know the filter and route around it. Robustness has to be designed in: training-time AT, runtime sensor fusion, monitoring-time anomaly detection.

> 🎓 **The intellectual takeaway.** Adversarial examples are not a bug in neural networks — they are a *feature* of how high-dimensional linear classifiers (and approximately-linear nonlinear classifiers) interact with imperceptible perturbations in `n`-dimensional space. Robustness is the engineering discipline of making those classifiers behave the way humans **expect** in those off-manifold regions. Most of M74 (LLM red-team) and this module (vision red-team) is a single intellectual program.

## ✅ Recap

- **White-box attacks (PGD, CW, AutoAttack)** are the gold standard for measurement; **black-box attacks (Square, HopSkipJump, NES, SignOPT, OptAttack)** are the deployment threat.
- **Physical / patch / shadow attacks** survive print + camera — the safety-critical concern.
- **Poisoning** (One-Shot-Kill, BadNets, Sleeper Agents) attacks the training set, not inference.
- **Adversarial training (Madry, TRADES, AWP)** is empirical SOTA; **randomized smoothing + diffusion purification** give provable / SOTA-post-hoc robustness.
- **Always evaluate with AutoAttack** — anything else risks gradient masking.
- **NLP adversarial (TextFooler, BERT-Attack)** is the same intellectual program as LLM red-teaming (M74) — discrete search over token space.
- **2026 production**: AT for safety-critical, ensemble + sensor-fusion + monitoring everywhere else.

Next: **M92 — Neural Style Transfer + AdaIN**.
